# 04 — RAGBench Results Matrix — Group 16

**Goal:** run our RAG pipeline over the 5 RAGBench domains under several **strategies** (configs), score every answer with the validated judge, and produce the **strategy × domain matrix** — the headline Task-1 deliverable.

**This is where everything comes together.** The pieces built brick-by-brick now run as one chain, once per question:

```
retrieve -> rerank -> repack -> prompt -> generate   (the pipeline answers)
       -> segment -> judge -> TRACe score            (the evaluator grades)
```

**An experiment = a config.** The `configs/*.yaml` files each define one pipeline setup. The runner loops `configs × domains`, writes one CSV row per pair = the matrix. The starter grid is a clean 2×2 that varies one lever at a time:

| | reranker `cross_encoder` | reranker `none` |
|---|---|---|
| **prompt `grounded`** | grounded_rerank | grounded_norerank |
| **prompt `minimal`** | minimal_rerank | minimal_norerank |

- **prompt** grounded vs minimal → the **Adherence** lever (does "answer only from context" curb hallucination?).
- **reranker** cross_encoder vs none → the **Relevance** lever (does re-scoring sharpen context?).

> **Prerequisite:** notebook 03 picked the judge. Set `JUDGE_MODEL` below to that winner.
> Runs on **Colab GPU**. The run is **resumable** — already-done `(config, domain)` pairs are skipped, so a disconnect won't lose finished work.

In [ ]:
# Colab setup: clone the repo + install dependencies (pipeline models + judge).
import os, sys
if not os.path.exists("src"):
    !git clone https://github.com/abhisagar123/CapstoneRAG.git
    os.chdir("CapstoneRAG")
sys.path.insert(0, os.getcwd())

!pip install -q datasets transformers accelerate bitsandbytes sentence-transformers faiss-cpu pyyaml nltk

## (optional) Hugging Face token

Only needed for the **hf/Colab** path with a **gated** model (e.g. Llama). The **local Ollama** path (default) needs no token.

**Safe setup:** add the token in Colab's **🔑 Secrets** panel (left sidebar) as `HF_TOKEN`, toggle *Notebook access* on. The cell below reads it from there — **never paste the token into a cell** (this repo is public; a committed token is a leaked credential).

In [ ]:
# (Colab only) Load HF_TOKEN from Secrets if present. Optional; not needed for the
# local Ollama path. Never hardcode a token here: this repo is public.
try:
    from google.colab import userdata
    import os
    _tok = userdata.get('HF_TOKEN')
    if _tok:
        os.environ['HF_TOKEN'] = _tok
        print('HF_TOKEN loaded from Colab Secrets.')
    else:
        print('No HF_TOKEN secret set — fine for the local Ollama path / ungated models.')
except Exception:
    print('Not on Colab — fine; the local Ollama path needs no token.')

## 1. Settings — what to run

- `JUDGE_MODEL` — the judge you chose in notebook 03 (non-Chinese; `llama3.1:8b` by default).
- `GEN_MODEL` — the **generator** (the model whose answers we are grading). Small for a first pass (`llama3.2:3b`); the configs name it too, but this overrides them so you set it in one place.
- `DOMAINS` — which of the 5 to run. Start with 1–2; add the rest once the flow is proven.
- `N` — examples per (config, domain). Small first: total judge calls = `len(configs) × len(DOMAINS) × N`.

In [ ]:
# --- Edit these, then run top-to-bottom ---
# Policy: open-source models may run LOCALLY, but NOT Chinese models (no Qwen).
BACKEND      = "ollama"                 # "ollama" (local Mac) or "hf" (Colab GPU)
JUDGE_MODEL  = "llama3.1:8b"            # the judge chosen in notebook 03 (NON-Chinese)
GEN_MODEL    = "llama3.2:3b"            # the generator under test (small first; NON-Chinese)
LOAD_IN_4BIT = True                     # (hf backend only) ignored by ollama

DOMAINS = ["GenKnowledge", "CustomerSupport"]   # start small; full set = 5 keys of DOMAIN_CONFIGS
N       = 10                                     # examples per (config, domain)

OUT_CSV = "results/ragbench_matrix.csv"

## 2. Register components + build the judge

`load_*()` registers the heavy components (real embedder / generator / reranker / judge) into the registry. Building the judge downloads + loads it onto the GPU (slow, one-time).

In [ ]:
import src
from src.embeddings import load_embedders
from src.generation import load_generators
from src.reranking import load_rerankers
from src.judge import load_judges
load_embedders(); load_generators(); load_rerankers(); load_judges()

from src.registry import build
if BACKEND == "ollama":
    judge = build("judge", "ollama", {"model": JUDGE_MODEL, "max_new_tokens": 1536})
else:
    judge = build("judge", "hf", {"model": JUDGE_MODEL, "load_in_4bit": LOAD_IN_4BIT,
                                  "max_new_tokens": 1536})
print(f"Judge ready: {JUDGE_MODEL}  (backend={BACKEND})")

## 3. Load the configs and build the (config × domain) grid

Each YAML is loaded, its `domain` is overridden per target domain, and its `generator.model` is set to `GEN_MODEL` (one knob, set above). We also attach the **filename** as `config_name` so every matrix row is traceable to its source file — and we guard against two configs collapsing to the same `config_id` (which would overwrite each other in the matrix).

In [ ]:
import glob, yaml
from src.runner import build_grid

raw_by_name = {os.path.basename(p): yaml.safe_load(open(p)) for p in sorted(glob.glob("configs/*.yaml"))}
print("config files:", list(raw_by_name))

# Point every config's generator at our chosen local/Colab model (backend-aware).
gen_override = ({"type": "ollama", "model": GEN_MODEL, "max_new_tokens": 256}
                if BACKEND == "ollama"
                else {"type": "hf", "model": GEN_MODEL, "load_in_4bit": LOAD_IN_4BIT})

# build_grid clones each config per domain + asserts no (config_id, domain) collision.
grid = build_grid(raw_by_name, DOMAINS, generator_override=gen_override)
print(f"built {len(grid)} (config x domain) pipelines, all distinct")

## 4. Run the matrix

`examples_for(cfg)` loads that config's domain (cached so each domain loads once). `run_matrix` runs each pipeline, segments + judges + scores every answer, and writes one row per `(config, domain)` to the CSV — flushing per row so a disconnect is safe.

We pass a small wrapper so each row also records `config_name`.

In [ ]:
from src.data_loader import load_domain
from src.segmentation import OutputSegmenter, RegexSplitter
from src.runner import run_named_matrix

SEG = OutputSegmenter(RegexSplitter())
_ex_cache = {}
def examples_for(cfg):
    if cfg.domain not in _ex_cache:
        _ex_cache[cfg.domain] = load_domain(cfg.domain, split="test", n=N, seed=42)
    return _ex_cache[cfg.domain]

# Shared with scripts/run_matrix.py — resumable + file-swap-safe (re-opens per row).
run_named_matrix(grid, examples_for, OUT_CSV, segmenter=SEG, judge=judge)

## 5. The matrix + a first read

Pivot the long CSV into the **strategy × domain** view, per metric. This is the table that goes in the report (alongside a comparison to the dataset's reference scores).

In [ ]:
import pandas as pd
df = pd.read_csv(OUT_CSV)
print("Raw rows:\n")
print(df[["config_name","domain","n","n_scored","relevance","utilization",
          "completeness","adherence","scoring"]].round(3).to_string(index=False))

print("\n\nStrategy × domain, per metric:")
for metric in ["relevance","utilization","completeness","adherence"]:
    print(f"\n--- {metric} ---")
    piv = df.pivot_table(index="config_name", columns="domain", values=metric)
    print(piv.round(3).to_string())

## 6. How to read it + next steps

**What the matrix shows.** Each cell is a strategy's mean TRACe score on a domain. Read it two ways:
- **Down a column** (fixed domain): which strategy wins *here*? Best-per-domain configs are exactly the per-domain "winning configs" the demo will use.
- **Across a row** (fixed strategy): how does one setup hold up across domains? Big swings = domain sensitivity (expected — legal ≠ finance).

**The two levers, isolated.**
- `grounded` vs `minimal` (same reranker) → the prompt's effect on **Adherence** (and Utilization).
- `cross_encoder` vs `none` (same prompt) → the reranker's effect on **Relevance**.

**Caveats (be honest in the report).**
- **Small `N`** — these are estimates to prove the flow + spot trends, not final numbers. Re-run the chosen winners at higher `N` for the cited table.
- **`corpus_mode: per_example`** — each question is answered against only its own documents (matches the reference setup). The *pooled* per-domain corpus is a separate Phase-2 experiment.
- **Chunking barely moves 8/12 sub-datasets** (they ship pre-segmented; see AI_CONTEXT §13) — so don't over-read chunking effects outside cuad/techqa/expertqa.
- **Compare to reference.** RAGBench ships reference TRACe scores; the report's key analysis is *our pipeline's* scores vs those, per domain — not just our configs against each other.

**Next:** pick the best config per domain, then either (a) scale `N` for final numbers, (b) add a chunking ablation on `cuad`, or (c) move to Task 2 (RGB).